# Lita Research Log Analysis

`research_logs/conversations/` と `research_logs/thoughts/` のCSVを読み込んで分析する。

In [ ]:
import pandas as pd
import glob
import json
from pathlib import Path

LOG_DIR = Path("research_logs")

# --- conversations ---
conv_files = sorted(glob.glob(str(LOG_DIR / "conversations" / "*.csv")))
if conv_files:
    conv = pd.concat([pd.read_csv(f) for f in conv_files], ignore_index=True)
    conv["timestamp"] = pd.to_datetime(conv["timestamp"])
    conv["metadata"] = conv["metadata"].apply(lambda x: json.loads(x) if pd.notna(x) and x else {})
    print(f"conversations: {len(conv)} rows from {len(conv_files)} files")
else:
    conv = pd.DataFrame()
    print("conversations: no data")

# --- thoughts ---
thought_files = sorted(glob.glob(str(LOG_DIR / "thoughts" / "*.csv")))
if thought_files:
    thoughts = pd.concat([pd.read_csv(f) for f in thought_files], ignore_index=True)
    thoughts["timestamp"] = pd.to_datetime(thoughts["timestamp"])
    thoughts["evaluation_details"] = thoughts["evaluation_details"].apply(
        lambda x: json.loads(x) if pd.notna(x) and x else {}
    )
    print(f"thoughts: {len(thoughts)} rows from {len(thought_files)} files")
else:
    thoughts = pd.DataFrame()
    print("thoughts: no data")

## 1. 基本統計

In [ ]:
if not conv.empty:
    counts = conv["event_type"].value_counts()
    total_turns = len(conv)
    user_msgs = counts.get("user_message", 0)
    ai_reactive = counts.get("ai_response", 0)
    ai_proactive = counts.get("proactive_intervention", 0)

    print(f"総イベント数:       {total_turns}")
    print(f"ユーザーメッセージ: {user_msgs}")
    print(f"AI反応的応答:       {ai_reactive}")
    print(f"AI自発的発言:       {ai_proactive}")
    print(f"Proactive率:        {ai_proactive / (ai_reactive + ai_proactive) * 100:.1f}%" if (ai_reactive + ai_proactive) > 0 else "")
    print()

    # セッション別
    print("--- セッション別 ---")
    for sid, g in conv.groupby("session_id"):
        c = g["event_type"].value_counts()
        duration = (g["timestamp"].max() - g["timestamp"].min()).total_seconds() / 60
        print(f"  {sid}: {len(g)} events, {duration:.0f}min, "
              f"user={c.get('user_message',0)}, reactive={c.get('ai_response',0)}, proactive={c.get('proactive_intervention',0)}")
else:
    print("No conversation data")

## 2. 応答時間分析

In [ ]:
if not conv.empty:
    conv_sorted = conv.sort_values("timestamp").reset_index(drop=True)

    # ユーザー発言 → AI応答の時間差
    response_times = []
    for i in range(len(conv_sorted) - 1):
        curr = conv_sorted.iloc[i]
        nxt = conv_sorted.iloc[i + 1]
        if curr["event_type"] == "user_message" and nxt["event_type"] == "ai_response":
            dt = (nxt["timestamp"] - curr["timestamp"]).total_seconds()
            response_times.append(dt)

    # AI発言 → ユーザー返答の時間差
    user_reply_times = []
    for i in range(len(conv_sorted) - 1):
        curr = conv_sorted.iloc[i]
        nxt = conv_sorted.iloc[i + 1]
        if curr["event_type"] in ("ai_response", "proactive_intervention") and nxt["event_type"] == "user_message":
            dt = (nxt["timestamp"] - curr["timestamp"]).total_seconds()
            user_reply_times.append(dt)

    if response_times:
        print(f"AI応答時間 (user→AI):  平均 {sum(response_times)/len(response_times):.1f}s, "
              f"中央値 {sorted(response_times)[len(response_times)//2]:.1f}s")
    if user_reply_times:
        print(f"ユーザー返答時間 (AI→user): 平均 {sum(user_reply_times)/len(user_reply_times):.1f}s, "
              f"中央値 {sorted(user_reply_times)[len(user_reply_times)//2]:.1f}s")
else:
    print("No data")

## 3. 介入受容率

Proactive発言の後にユーザーが返事したかどうか。

In [ ]:
if not conv.empty:
    conv_sorted = conv.sort_values("timestamp").reset_index(drop=True)
    proactive_indices = conv_sorted[conv_sorted["event_type"] == "proactive_intervention"].index

    accepted = 0
    ignored = 0
    for idx in proactive_indices:
        if idx + 1 < len(conv_sorted):
            nxt = conv_sorted.iloc[idx + 1]
            if nxt["event_type"] == "user_message":
                accepted += 1
            else:
                ignored += 1
        else:
            ignored += 1  # 最後のメッセージ = 返答なし

    total = accepted + ignored
    if total > 0:
        print(f"Proactive発言数: {total}")
        print(f"ユーザーが返答:  {accepted} ({accepted/total*100:.0f}%)")
        print(f"返答なし:        {ignored} ({ignored/total*100:.0f}%)")
    else:
        print("Proactive発言なし")
else:
    print("No data")

## 4. 思考ログ分析

思考の生成・評価・発言の統計。`thoughts/` CSVにデータが溜まってから使える。

In [ ]:
if not thoughts.empty:
    total = len(thoughts)
    expressed = thoughts["was_expressed"].sum()
    scores = thoughts["motivation_score"]

    print(f"生成された思考数:   {total}")
    print(f"発言された思考:     {expressed} ({expressed/total*100:.0f}%)")
    print(f"発言されなかった:   {total - expressed} ({(total-expressed)/total*100:.0f}%)")
    print()
    print(f"Motivation Score:")
    print(f"  平均:   {scores.mean():.2f}")
    print(f"  中央値: {scores.median():.2f}")
    print(f"  最小:   {scores.min():.2f}")
    print(f"  最大:   {scores.max():.2f}")
    print()

    # トリガー別の内訳
    print("--- トリガー別 ---")
    for trigger, g in thoughts.groupby("trigger_reason"):
        e = g["was_expressed"].sum()
        print(f"  {trigger}: {len(g)} thoughts, {e} expressed, avg_score={g['motivation_score'].mean():.2f}")
else:
    print("No thought data yet (thoughts/ is empty)")

## 5. 会話内容のプレビュー

直近の会話を見やすく表示。

In [ ]:
if not conv.empty:
    conv_sorted = conv.sort_values("timestamp").reset_index(drop=True)

    # 直近N件を表示
    N = 30
    recent = conv_sorted.tail(N)

    for _, row in recent.iterrows():
        ts = row["timestamp"].strftime("%m/%d %H:%M")
        et = row["event_type"]
        content = row["content"][:80]

        if et == "user_message":
            label = "\033[94muser\033[0m"  # blue
        elif et == "ai_response":
            label = "\033[92mLita\033[0m"  # green
        else:
            label = "\033[93mLita(proactive)\033[0m"  # yellow

        print(f"[{ts}] {label}: {content}")
else:
    print("No data")